The car routes include congested times.  For each TBI record, find the corresponding time, and match to the appropriate routes for that time. 

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd

import keyring

In [ ]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [ ]:
# This is used to avoid hard-coding directories. 
# To set the directory, use the command prompt or a notebook you don't check in.  run:
# import keyring
# keyring.set_password("msp", "vmt_reduction_dir", <directory>)

# get base path for data
data_dir = keyring.get_password("msp", "vmt_reduction_dir")

In [ ]:
def get_car_congestion_period(row):
    hour = int(row["arrive_time"][0:2])
    sunday = row["travel_dow"] == "Sunday"
    saturday = row["travel_dow"] == "Saturday"
    if sunday:
        query = "sundays"
    elif saturday:
        query = "saturdays_"
    else:
        query = "weekdays_"
    
    if hour >= 0 and hour <= 5:
        query += "0-6"
    elif hour >= 20 and hour <= 23:
        query += "20-24"
    else:
        query += str(hour) + "-" + str(hour + 1)

    return query

In [ ]:
# read in the data
tbi = pd.read_csv(data_dir + "/data_processed/tbi_cleaned.csv")

In [ ]:
tbi['congestion_period'] = tbi.apply(get_car_congestion_period, axis=1)

In [ ]:
# select the appropriate rows from the congested car data
periods = tbi['congestion_period'].unique()

for period in periods:
    selected_trips = tbi[tbi['congestion_period']==period]    
    selected_trip_ids = selected_trips['trip_id']
    
    print ('Processing ' + period + ' with ' + str(len(selected_trip_ids)) + ' trips.')
    
    all_routes = gpd.read_file(data_dir + "/Data_Processed/car-congestion-1d39c/car_" + period + '.gpkg')
    
    selected_routes = all_routes.merge(selected_trip_ids, on='trip_id', how='inner')
    selected_routes.to_parquet(data_dir + "/Data_processed/car-congestion-1d39c/selected_car_" + period + ".parquet", index=False)

In [ ]:
# now merge the files
periods = tbi['congestion_period'].unique()

gdf = gpd.GeoDataFrame()
for period in periods:
    gdf_period = gpd.read_parquet(data_dir + "/Data_processed/car_congestion/selected_car_" + period + ".parquet")
    gdf = pd.concat([gdf, gdf_period])
    
gdf.to_parquet(data_dir + "/Data_processed/geodata/car_congestion.parquet", index=False)